In [27]:
!pip install -q transformers timm accelerate sentencepiece

In [28]:
# ==========================================
# CELL 4: FIXED DATA LOADER (Only reads Filtered_VQA files)
# ==========================================
import glob
import json
import random
from tqdm import tqdm
from collections import Counter

def load_vqa_data():
    """Load VQA data from Filtered_VQA_*.jsonl files only"""
    all_samples = []

    # ONLY get Filtered_VQA files (not Vocabulary files)
    jsonl_files = glob.glob(os.path.join(cfg.DATA_DIR, "Filtered_VQA_*.jsonl"))

    print(f"Found {len(jsonl_files)} Filtered_VQA files")

    if not jsonl_files:
        print("No Filtered_VQA files found!")
        print("Looking for any .jsonl files...")
        jsonl_files = glob.glob(os.path.join(cfg.DATA_DIR, "*.jsonl"))
        print(f"Found {len(jsonl_files)} total .jsonl files")

    if not jsonl_files:
        return []

    # Build image cache
    print("Building image cache...")
    image_cache = {}
    try:
        for f in os.listdir(cfg.IMAGE_DIR):
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')):
                # Store with and without unsplash_ prefix
                img_id = os.path.splitext(f)[0]
                clean_id = img_id.replace('unsplash_', '')
                image_cache[img_id] = os.path.join(cfg.IMAGE_DIR, f)
                image_cache[clean_id] = os.path.join(cfg.IMAGE_DIR, f)
        print(f"Found {len(image_cache)//2} images")
    except OSError as e:
        print(f"Cannot list images: {e}")
        return []

    # Load each Filtered_VQA file
    for file_path in tqdm(jsonl_files, desc="Loading data"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line_num, line in enumerate(f):
                    if not line.strip():
                        continue
                    try:
                        item = json.loads(line.strip())

                        # Get image_id
                        image_id = item.get('image_id')
                        if not image_id:
                            continue

                        # Clean image_id (remove unsplash_ if present)
                        image_id = str(image_id).replace('unsplash_', '')

                        # Find image path
                        image_path = image_cache.get(image_id)
                        if not image_path:
                            # Try with unsplash_ prefix
                            image_path = image_cache.get('unsplash_' + image_id)

                        if not image_path:
                            continue

                        # Get QA pairs
                        qa_pairs = item.get('qa_pairs', [])

                        for qa in qa_pairs:
                            question = qa.get('question', '')
                            answer = qa.get('answer', '')

                            if question and answer:
                                all_samples.append({
                                    'image_path': image_path,
                                    'question': question.strip(),
                                    'answer': answer.strip().lower()
                                })
                    except json.JSONDecodeError as e:
                        print(f"  JSON error line {line_num}: {e}")
                        continue
        except Exception as e:
            print(f"Error reading {os.path.basename(file_path)}: {e}")

    print(f"\n✅ Loaded {len(all_samples)} Q&A pairs")
    return all_samples

# Load data
print("\n" + "="*60)
print("LOADING VQA DATA")
print("="*60)

all_samples = load_vqa_data()

if len(all_samples) > 0:
    print(f"\n📊 Sample:")
    print(f"   Image: {all_samples[0]['image_path']}")
    print(f"   Question: {all_samples[0]['question'][:80]}")
    print(f"   Answer: {all_samples[0]['answer']}")

    # Show answer distribution
    answers = [s['answer'] for s in all_samples]
    print(f"\n📊 Answer distribution (top 15):")
    for ans, count in Counter(answers).most_common(15):
        print(f"   {ans}: {count}")
else:
    print("\n❌ No data loaded!")

    # Debug: Show first line of first file
    print("\n🔍 Debug: Reading first Filtered_VQA file directly...")
    test_files = glob.glob(os.path.join(cfg.DATA_DIR, "Filtered_VQA_*.jsonl"))
    if test_files:
        with open(test_files[0], 'r') as f:
            first_line = f.readline()
            print(f"First line: {first_line[:500]}")


LOADING VQA DATA
Found 18 Filtered_VQA files
Building image cache...
Found 44804 images


Loading data: 100%|██████████| 18/18 [00:00<00:00, 75.25it/s]


✅ Loaded 18050 Q&A pairs

📊 Sample:
   Image: /content/drive/MyDrive/Deep Learning Project/unzipped_images/raw/pexels_569679.jpg
   Question: What type of scene?
   Answer: city street

📊 Answer distribution (top 15):
   green: 641
   red: 538
   sunny: 495
   blue: 476
   calm: 473
   two: 460
   no: 453
   white: 451
   several: 439
   wood: 417
   one: 411
   outdoor: 374
   indoor: 373
   market: 371
   daytime: 325


In [29]:
import os
import json
import random
import numpy as np
from PIL import Image
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import (
    AutoTokenizer,
    AutoModel,
    ViTModel,
    ViTImageProcessor,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [31]:
class CFG:
    # ===== YOUR CORRECT PATHS =====
    DATA_DIR = "/content/drive/MyDrive/Deep Learning Project/2 words Q&As/New Q&As/Filtered Q&As"
    IMAGE_DIR = "/content/drive/MyDrive/Deep Learning Project/unzipped_images/raw"
    CHECKPOINT_DIR = "/content/drive/MyDrive/vqa_checkpoints"

    # Model names
    MODEL_NAME = 'bert-base-uncased'
    VIT_NAME = 'google/vit-base-patch16-224'

    # Data params
    MAX_LEN = 64
    IMAGE_SIZE = 224

    # Training params
    BATCH_SIZE = 16
    EPOCHS = 12
    LR = 3e-5
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1
    VAL_SPLIT = 0.15
    SEED = 42

    # Model architecture
    HIDDEN_DIM = 768
    NUM_HEADS = 8
    NUM_FUSION_LAYERS = 4  # 4-layer bidirectional fusion
    DROPOUT = 0.1

    # Other
    NUM_WORKERS = 2
    AMP = False

    # Create checkpoint dir
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

cfg = CFG()
print(f"✅ Config loaded")
print(f"   DATA_DIR: {cfg.DATA_DIR}")
print(f"   IMAGE_DIR: {cfg.IMAGE_DIR}")
print(f"   Checkpoint dir: {cfg.CHECKPOINT_DIR}")

✅ Config loaded
   DATA_DIR: /content/drive/MyDrive/Deep Learning Project/2 words Q&As/New Q&As/Filtered Q&As
   IMAGE_DIR: /content/drive/MyDrive/Deep Learning Project/unzipped_images/raw
   Checkpoint dir: /content/drive/MyDrive/vqa_checkpoints


In [32]:
import glob
from tqdm import tqdm

def load_vqa_data():
    """Load VQA data from your JSON/JSONL files"""
    all_samples = []

    # Get all JSON and JSONL files
    json_files = glob.glob(os.path.join(cfg.DATA_DIR, "*.json"))
    jsonl_files = glob.glob(os.path.join(cfg.DATA_DIR, "*.jsonl"))
    all_files = json_files + jsonl_files

    print(f"Found {len(all_files)} files")

    if not all_files:
        print("No files found!")
        return []

    # Build image cache
    print("Building image cache...")
    image_cache = {}
    try:
        for f in os.listdir(cfg.IMAGE_DIR):
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')):
                img_id = os.path.splitext(f)[0]
                # Store with both original and cleaned ID
                clean_id = img_id.replace('unsplash_', '')
                image_cache[img_id] = os.path.join(cfg.IMAGE_DIR, f)
                image_cache[clean_id] = os.path.join(cfg.IMAGE_DIR, f)
        print(f"Found {len(image_cache)//2} images")
    except OSError as e:
        print(f"Cannot list images: {e}")
        return []

    # Load each file
    for file_path in tqdm(all_files, desc="Loading data"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()

                # Try to parse as JSON array first
                try:
                    data_list = json.loads(content)
                    if isinstance(data_list, list):
                        items = data_list
                    else:
                        items = [data_list]
                except:
                    # Try line-by-line JSONL
                    items = []
                    for line in content.split('\n'):
                        if line.strip():
                            try:
                                items.append(json.loads(line))
                            except:
                                pass

                for item in items:
                    # Handle different possible field names
                    image_id = item.get('image_id') or item.get('id') or item.get('imageId')
                    if not image_id:
                        continue

                    image_id = str(image_id).replace('unsplash_', '')

                    # Find image path
                    image_path = None
                    for key in [image_id, 'unsplash_' + image_id]:
                        if key in image_cache:
                            image_path = image_cache[key]
                            break

                    if not image_path:
                        continue

                    # Get QA pairs
                    qa_pairs = item.get('qa_pairs') or item.get('qas') or item.get('questions') or []

                    for qa in qa_pairs:
                        question = qa.get('question') or qa.get('q') or ''
                        answer = qa.get('answer') or qa.get('a') or ''

                        if question and answer:
                            all_samples.append({
                                'image_path': image_path,
                                'question': question.strip(),
                                'answer': answer.strip().lower()
                            })
        except Exception as e:
            print(f"Error reading {os.path.basename(file_path)}: {e}")

    print(f"\n✅ Loaded {len(all_samples)} Q&A pairs")
    return all_samples

# Load data
print("\n" + "="*60)
print("LOADING VQA DATA")
print("="*60)

all_samples = load_vqa_data()

if len(all_samples) > 0:
    print(f"\n📊 Sample:")
    print(f"   Image: {all_samples[0]['image_path']}")
    print(f"   Question: {all_samples[0]['question'][:80]}")
    print(f"   Answer: {all_samples[0]['answer']}")

    # Show answer distribution
    answers = [s['answer'] for s in all_samples]
    print(f"\n📊 Top 10 answers:")
    for ans, count in Counter(answers).most_common(10):
        print(f"   {ans}: {count}")
else:
    print("\n❌ No data loaded!")


LOADING VQA DATA
Found 35 files
Building image cache...
Found 44804 images


Loading data: 100%|██████████| 35/35 [00:00<00:00, 102.76it/s]


✅ Loaded 18050 Q&A pairs

📊 Sample:
   Image: /content/drive/MyDrive/Deep Learning Project/unzipped_images/raw/pexels_569679.jpg
   Question: What type of scene?
   Answer: city street

📊 Top 10 answers:
   green: 641
   red: 538
   sunny: 495
   blue: 476
   calm: 473
   two: 460
   no: 453
   white: 451
   several: 439
   wood: 417


In [33]:
MAX_ANSWERS = 1000

answer_counter = Counter([x['answer'].lower().strip() for x in all_samples])

most_common_answers = answer_counter.most_common(MAX_ANSWERS)

answer2id = {
    ans: idx
    for idx, (ans, _) in enumerate(most_common_answers)
}

id2answer = {
    idx: ans
    for ans, idx in answer2id.items()
}

# Filter samples to only keep frequent answers
filtered_samples = []
for s in all_samples:
    ans = s['answer'].lower().strip()
    if ans in answer2id:
        filtered_samples.append(s)

print('Filtered samples:', len(filtered_samples))
print('Num answers:', len(answer2id))
print(f'Top 10 answers: {list(answer2id.keys())[:10]}')

Filtered samples: 18050
Num answers: 386
Top 10 answers: ['green', 'red', 'sunny', 'blue', 'calm', 'two', 'no', 'white', 'several', 'wood']


In [34]:
# Split by image to avoid leakage
unique_images = list(set(s['image_path'] for s in filtered_samples))
random.seed(cfg.SEED)
random.shuffle(unique_images)

n_val = int(len(unique_images) * cfg.VAL_SPLIT)
val_images = set(unique_images[:n_val])
train_images = set(unique_images[n_val:])

train_samples = [s for s in filtered_samples if s['image_path'] not in val_images]
val_samples = [s for s in filtered_samples if s['image_path'] in val_images]

print(f'Train: {len(train_samples)} samples from {len(train_images)} images')
print(f'Val: {len(val_samples)} samples from {len(val_images)} images')

Train: 15339 samples from 2817 images
Val: 2711 samples from 497 images


In [35]:
tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_NAME)
image_processor = ViTImageProcessor.from_pretrained(cfg.VIT_NAME)

# Transforms
train_transforms = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
])

val_transforms = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
])

In [36]:
class VQADataset(Dataset):
    def __init__(self, samples, transforms=None, augment=False):
        self.samples = samples
        self.transforms = transforms
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        image_path = sample['image_path']
        question = sample['question']
        answer = sample['answer'].lower().strip()

        image = Image.open(image_path).convert('RGB')

        if self.transforms:
            image = self.transforms(image)

        # Process image
        pixel_values = image_processor(images=image, return_tensors='pt')['pixel_values'][0]

        # Process question
        encoded = tokenizer(
            question,
            padding='max_length',
            truncation=True,
            max_length=cfg.MAX_LEN,
            return_tensors='pt'
        )

        return {
            'pixel_values': pixel_values,
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'label': torch.tensor(answer2id[answer], dtype=torch.long)
        }

In [37]:
train_dataset = VQADataset(train_samples, train_transforms, augment=True)
val_dataset = VQADataset(val_samples, val_transforms, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 959, Val batches: 170


In [38]:
class CrossAttentionBlock(nn.Module):
    """Cross-attention block with residual connections and FFN"""
    def __init__(self, dim, num_heads, dropout=0.1):
        super().__init__()

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(dim)

        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim)
        )

        self.norm2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value):
        attended, _ = self.cross_attn(
            query=query,
            key=key_value,
            value=key_value
        )

        x = self.norm1(query + self.dropout(attended))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x


class AttentionPooling(nn.Module):
    """Attention-based pooling over sequence dimension"""
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Linear(dim, 1)

    def forward(self, x, mask):
        weights = self.attn(x)
        mask = mask.unsqueeze(-1)
        weights = weights.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(weights, dim=1)
        pooled = (weights * x).sum(dim=1)
        return pooled

In [39]:
class VQAModel(nn.Module):
    def __init__(self, num_answers):
        super().__init__()

        # Vision encoder
        self.vit = ViTModel.from_pretrained(cfg.VIT_NAME)
        self.vit.gradient_checkpointing_enable()

        # Text encoder
        self.text_encoder = AutoModel.from_pretrained(cfg.MODEL_NAME)

        # Freeze embeddings initially
        for p in self.text_encoder.embeddings.parameters():
            p.requires_grad = False

        # Projection layers
        self.img_proj = nn.Linear(cfg.HIDDEN_DIM, cfg.HIDDEN_DIM)
        self.txt_proj = nn.Linear(cfg.HIDDEN_DIM, cfg.HIDDEN_DIM)

        # 4-layer BIDIRECTIONAL cross-attention fusion
        self.text_to_image = nn.ModuleList([
            CrossAttentionBlock(cfg.HIDDEN_DIM, cfg.NUM_HEADS, cfg.DROPOUT)
            for _ in range(cfg.NUM_FUSION_LAYERS)
        ])

        self.image_to_text = nn.ModuleList([
            CrossAttentionBlock(cfg.HIDDEN_DIM, cfg.NUM_HEADS, cfg.DROPOUT)
            for _ in range(cfg.NUM_FUSION_LAYERS)
        ])

        # Pooling and classifier
        self.pool = AttentionPooling(cfg.HIDDEN_DIM)

        self.classifier = nn.Sequential(
            nn.Linear(cfg.HIDDEN_DIM, cfg.HIDDEN_DIM),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(cfg.HIDDEN_DIM, num_answers)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        # Vision: get all patches (remove CLS token)
        vit_out = self.vit(pixel_values=pixel_values)
        img_feat = vit_out.last_hidden_state[:, 1:, :]
        img_feat = self.img_proj(img_feat)

        # Text: get all token embeddings
        txt_out = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        txt_feat = txt_out.last_hidden_state
        txt_feat = self.txt_proj(txt_feat)

        # BIDIRECTIONAL FUSION: 4 layers
        for t2i, i2t in zip(self.text_to_image, self.image_to_text):
            txt_feat = t2i(txt_feat, img_feat)
            img_feat = i2t(img_feat, txt_feat)

        # Pool text features
        pooled = self.pool(txt_feat, attention_mask)

        # Classify
        logits = self.classifier(pooled)

        return logits

In [40]:
model = VQAModel(num_answers=len(answer2id))
model = model.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params: {total_params/1e6:.1f}M")
print(f"Trainable params: {trainable_params/1e6:.1f}M")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total params: 254.6M
Trainable params: 230.8M


In [41]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.LR,
    weight_decay=cfg.WEIGHT_DECAY
)

num_training_steps = len(train_loader) * cfg.EPOCHS
warmup_steps = int(num_training_steps * cfg.WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=num_training_steps
)

scaler = torch.cuda.amp.GradScaler(enabled=cfg.AMP)

checkpoint_path = os.path.join(cfg.CHECKPOINT_DIR, 'vqa_4layer_fusion.pt')
print(f"Checkpoint will be saved to: {checkpoint_path}")

Checkpoint will be saved to: /content/drive/MyDrive/vqa_checkpoints/vqa_4layer_fusion.pt


/tmp/ipykernel_6384/2255615552.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg.AMP)


In [42]:
# ==========================================
# CELL 25: FIXED TRAINING FUNCTIONS (With Safe AMP)
# ==========================================
def train_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc='Training', leave=False)
    for batch in pbar:
        pixel_values = batch['pixel_values'].to(DEVICE)
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        optimizer.zero_grad()

        # Use autocast with explicit dtype or disable for first epoch
        if cfg.AMP and epoch > 0:  # Only use AMP after first epoch to avoid overflow
            with torch.cuda.amp.autocast():
                logits = model(pixel_values, input_ids, attention_mask)
                loss = criterion(logits, labels)
        else:
            logits = model(pixel_values, input_ids, attention_mask)
            loss = criterion(logits, labels)

        if cfg.AMP and epoch > 0:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        scheduler.step()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        correct += (logits.argmax(1) == labels).sum().item()
        total += bs

        pbar.set_postfix({'loss': f'{loss.item():.3f}'})

    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc='Validation', leave=False):
        pixel_values = batch['pixel_values'].to(DEVICE)
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        if cfg.AMP:
            with torch.cuda.amp.autocast():
                logits = model(pixel_values, input_ids, attention_mask)
                loss = criterion(logits, labels)
        else:
            logits = model(pixel_values, input_ids, attention_mask)
            loss = criterion(logits, labels)

        bs = labels.size(0)
        total_loss += loss.item() * bs
        correct += (logits.argmax(1) == labels).sum().item()
        total += bs

    return total_loss / total, correct / total

In [ ]:
best_val_acc = 0
history = []

for epoch in range(1, cfg.EPOCHS + 1):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch}/{cfg.EPOCHS}")
    print(f"{'='*60}")

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, scaler)
    val_loss, val_acc = eval_epoch(model, val_loader)

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc
    })

    print(f"\n📊 Results:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'train_acc': train_acc,
            'answer2id': answer2id,
            'id2answer': id2answer,
            'history': history,
        }, checkpoint_path)
        print(f"   💾 Saved BEST model (acc: {val_acc:.4f})")

    # Save checkpoint every epoch
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'val_acc': val_acc,
    }, os.path.join(cfg.CHECKPOINT_DIR, f'checkpoint_epoch_{epoch}.pt'))

print(f"\n{'='*60}")
print(f"🏆 Training Complete!")
print(f"   Best Validation Accuracy: {best_val_acc:.4f} ({best_val_acc:.2%})")
print(f"   Model saved to: {checkpoint_path}")
print(f"{'='*60}")


Epoch 1/12


Validation:  58%|█████▊    | 99/170 [01:19<01:20,  1.14s/it]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



📊 Results:
   Train Loss: 4.0871 | Train Acc: 0.2465
   Val Loss:   2.5975 | Val Acc:   0.4139
   💾 Saved BEST model (acc: 0.4139)

Epoch 2/12


Validation:  56%|█████▌    | 95/170 [00:40<00:28,  2.68it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



📊 Results:
   Train Loss: 2.2163 | Train Acc: 0.4578
   Val Loss:   1.9401 | Val Acc:   0.4950
   💾 Saved BEST model (acc: 0.4950)

Epoch 3/12


...

Validation:  56%|█████▋    | 96/170 [00:39<00:27,  2.70it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



📊 Results:
   Train Loss: 1.3975 | Train Acc: 0.5925
   Val Loss:   1.6132 | Val Acc:   0.5338
   💾 Saved BEST model (acc: 0.5338)

Epoch 5/12


Validation:  57%|█████▋    | 97/170 [00:40<00:29,  2.46it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



📊 Results:
   Train Loss: 1.1452 | Train Acc: 0.6570
   Val Loss:   1.5773 | Val Acc:   0.5448
   💾 Saved BEST model (acc: 0.5448)

Epoch 6/12


Training:  42%|████▏     | 400/959 [09:36<11:17,  1.21s/it, loss=0.487]

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot([h['train_loss'] for h in history], label='Train Loss', linewidth=2)
axes[0].plot([h['val_loss'] for h in history], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot([h['train_acc'] for h in history], label='Train Acc', linewidth=2)
axes[1].plot([h['val_acc'] for h in history], label='Val Acc', linewidth=2)
axes[1].axhline(y=best_val_acc, color='green', linestyle='--', label=f'Best: {best_val_acc:.2%}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.CHECKPOINT_DIR, 'training_curves_4layer.png'), dpi=150)
plt.show()

In [ ]:
from IPython.display import display, clear_output, Image as IPImage
import ipywidgets as widgets

@torch.no_grad()
def predict_single(image_path, question, model, top_k=5):
    model.eval()

    image = Image.open(image_path).convert('RGB')
    image = val_transforms(image)
    pixel_values = image_processor(images=image, return_tensors='pt')['pixel_values'].to(DEVICE)

    encoded = tokenizer(
        question,
        padding='max_length',
        truncation=True,
        max_length=cfg.MAX_LEN,
        return_tensors='pt'
    )
    input_ids = encoded['input_ids'].to(DEVICE)
    attention_mask = encoded['attention_mask'].to(DEVICE)

    logits = model(pixel_values, input_ids, attention_mask)
    probs = F.softmax(logits, dim=-1).squeeze(0)

    top_probs, top_idxs = probs.topk(top_k)
    results = [(id2answer[i.item()], p.item()) for i, p in zip(top_idxs, top_probs)]

    return results

# Load best model
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f"✅ Loaded model from epoch {checkpoint['epoch']} (val_acc: {checkpoint['val_acc']:.2%})")

    print("\n" + "="*60)
    print("🎯 VQA Inference Demo Ready! (4-Layer Bidirectional Fusion)")
    print("="*60)

    uploader = widgets.FileUpload(accept='image/*', multiple=False, description='📁 Upload Image')
    question_input = widgets.Text(placeholder="Ask a question about the image...", description="❓ Question:")
    predict_button = widgets.Button(description="🔍 Predict", button_style='success')
    output = widgets.Output()

    def on_predict(b):
        with output:
            clear_output()
            if not uploader.value:
                print("⚠️ Please upload an image first")
                return
            if not question_input.value:
                print("⚠️ Please enter a question")
                return

            uploaded = list(uploader.value.values())[0]
            img_path = "/tmp/temp_image.jpg"
            with open(img_path, 'wb') as f:
                f.write(uploaded['content'])

            display(IPImage(img_path, width=300))
            results = predict_single(img_path, question_input.value, model)

            print(f"\n❓ Question: {question_input.value}")
            print("\n🏆 Top Predictions:")
            medals = ['🥇', '🥈', '🥉', '4️⃣', '5️⃣']
            for i, (label, prob) in enumerate(results):
                bar = '█' * int(prob * 30)
                print(f"   {medals[i]} {label:<25} {prob:.1%} {bar}")

    predict_button.on_click(on_predict)

    display(widgets.VBox([
        widgets.HTML("<h3>📷 VQA Model - 4-Layer Bidirectional Fusion</h3>"),
        uploader,
        question_input,
        predict_button,
        output
    ]))
else:
    print(f"⚠️ No checkpoint found. Please train first (run Cell 15)")